<a href="https://colab.research.google.com/github/newboo-me/Amazon-Product-Reviews-Sentiment-Analysis/blob/main/Amazon_Product_Reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#  อัปโหลดไฟล์ kaggle.json เข้า Colab
from google.colab import files

files.upload()

#  ย้ายไฟล์ไปยังโฟลเดอร์ของระบบ และตั้งสิทธิ์
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

#  คำสั่งดาวน์โหลด Dataset (ก๊อปปี้ API Command จากหน้าเว็บ Kaggle มาวาง)
#  ดาวน์โหลดชุดข้อมูล Amazon Product Reviews
!kaggle datasets download -d arhamrumi/amazon-product-reviews

#  แตกไฟล์ zip ออกมาใช้งาน
!unzip amazon-product-reviews.zip

#  อ่านไฟล์เข้า Pandas ได้ทันที
import pandas as pd

# ตรวจสอบชื่อไฟล์ที่แตกออกมา (มักจะเป็น Reviews.csv)
df = pd.read_csv('Reviews.csv')
df.head()

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/arhamrumi/amazon-product-reviews
License(s): CC0-1.0
100% 115M/115M [00:01<00:00, 93.1MB/s]

Archive:  amazon-product-reviews.zip
  inflating: Reviews.csv             


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


| คอลัมน์ | ความหมาย | บทบาทในงาน NLP & Analytics / การจัดการ |
| :--- | :--- | :--- |
| **Id** | รหัสลำดับแถว (Running Number) | ไม่มีความหมายทางสถิติ ตัดทิ้ง (Drop)  |
| **ProductId** | รหัสสินค้าของ Amazon | ใช้หา Insight สินค้าที่ได้คะแนนเฉลี่ยสูงสุด หรือถูกรีวิวมากที่สุด |
| **UserId** | รหัสประจำตัวของผู้รีวิว | ใช้ตรวจสอบผู้ใช้ที่รีวิวบ่อยผิดปกติ  |
| **ProfileName** | ชื่อที่แสดงของผู้รีวิว (ข้อมูลระบุตัวตน) | ตัดทิ้งได้ |
| **HelpfulnessNumerator** | จำนวนคนที่กดโหวตว่ารีวิวนี้ "มีประโยชน์" | ใช้คำนวณความน่าเชื่อถือของรีวิว |
| **HelpfulnessDenominator** | จำนวนคนที่กดโหวตทั้งหมด | ใช้เป็นตัวหารเพื่อหาอัตราส่วนคะแนนรีวิว |
| **Score** | คะแนนความพึงพอใจ (1 ถึง 5 ดาว) | **ตัวแปรเป้าหมาย (Target - y)**: แปลงเป็น Sentiment (บวก/ลบ) |
| **Time** | วันเวลาที่รีวิว (Unix Timestamp) | แปลงเป็น Datetime เพื่อดูแนวโน้มคะแนนตามช่วงเวลา |
| **Summary** | สรุปใจความสั้นๆ ของรีวิว | นำไปรวมเข้ากับ Text  |
| **Text** | เนื้อหารีวิวฉบับเต็มของลูกค้า | **ตัวแปรหลัก (Feature - X)**:  |

**Helpfulness Ratio**:เพื่อดูว่ารีวิวที่มีคนเห็นด้วยเยอะ มีทิศทางอารมณ์โน้มเอียงไปทางบวกหรือลบมากกว่ากัน
**Review Length**: นับจำนวนคำและความยาวของข้อความรีวิว

In [ ]:
import numpy as np
import pandas as pd

#  กรองรีวิวที่เป็นกลาง (Score = 3) ออกไปก่อน
df_filtered = df[df['Score'] != 3].copy()

#  แปลง Score(คะเเนนรีวิว) เป็น Sentiment(1 = Positive, 0 = Negative)
df_filtered['Sentiment'] = np.where(df_filtered['Score'] > 3, 1, 0)

#  จัดการเวลา date เป็น Datetime
df_filtered['Date'] = pd.to_datetime(df_filtered['Time'], unit='s')

#  รวม Summary และ Text เข้าด้วยกันเป็นฟีเจอร์ข้อความเดียว
df_filtered['Clean_Text'] = (
    df_filtered['Summary'].fillna('') + ' ' + df_filtered['Text'].fillna('')
)

#  สุ่มตัวอย่าง 50,000 แถวเพื่อความรวดเร็วในการประมวลผล NLP ใช้ การวิเคราะห์ความรู้สึก (Sentiment Analysis): จำแนกข้อความรีวิวหรือความคิดเห็นว่าเป็นเชิงบวก เป็นกลาง หรือเชิงลบ
df_sampled = df_filtered.sample(n=50000, random_state=42).reset_index(drop=True)

print("สัดส่วน Sentiment ในชุดข้อมูลตัวอย่าง:")
print(df_sampled['Sentiment'].value_counts(normalize=True))

สัดส่วน Sentiment ในชุดข้อมูลตัวอย่าง:
Sentiment
1    0.84294
0    0.15706
Name: proportion, dtype: float64


**รีวิว 3 ดาว**ตัดออกกันไม่ให้โมเดลสับสนจากคำศัพท์ที่ขัดแย้งกันในข้อความเดียว
**ลดความซับซ้อนให้ตรงโจทย์ธุรกิจ** เเปลง คะแนน 1 ถึง 2 ดาว--->Negative (0): ลูกค้าไม่พอใจ
เเปลงคะแนน 4 ถึง 5 ดาว--->Positive (1): ลูกค้าพึงพอใจ

In [ ]:
import re
import nltk
from nltk.corpus import stopwords

# ดาวน์โหลดรายการ Stopwords ภาษาอังกฤษ
nltk.download('stopwords')

# กำหนดชุดคำ Stopwords และยกเว้นคำปฏิเสธเพื่อรักษาความหมายเชิงลบ เพื่อไม่ให้สูญเสียบริบทเชิงลบ
stop_words = set(stopwords.words('english'))
negation_words = {'not', 'no', 'nor', 'neither', 'never'}
stop_words = stop_words - negation_words

def clean_review_text(text):# คอลัมtext
    if not isinstance(text, str):
        return ""

    # ลบ HTML tags
    text = re.sub(r'<.*?>', ' ', text)

    # แปลงเป็นตัวพิมพ์เล็ก
    text = text.lower()

    # ลบอักขระพิเศษ ตัวเลข และเครื่องหมายวรรคตอน (เก็บเฉพาะตัวอักษร a-z)
    text = re.sub(r'[^a-z\s]', ' ', text)

    # ตัดคำและคัดกรอง Stopwords (ตัดคำฟุ่มเฟือย ไม่มีความหมายเชิงอารมณ์ออก เช่น the, is, atเเละตัดคำที่สั้นกว่า 2 ตัวอักษรออกด้วย
    # โดยยกเว้นคำปฏิเสธบางคำไว้ (เช่น not, no) เพื่อไม่ให้สูญเสียบริบทเชิงลบ)
    tokens = text.split()
    cleaned_tokens = [word for word in tokens if word not in stop_words and len(word) > 1]

    return " ".join(cleaned_tokens)

# นำฟังก์ชันไปประมวลผลข้อความทั้งหมดในชุดข้อมูลตัวอย่าง
print("กำลังทำความสะอาดข้อความ...")
df_sampled['Processed_Text'] = df_sampled['Clean_Text'].apply(clean_review_text)
print("ทำความสะอาดข้อความเสร็จสิ้น!")

# ตรวจสอบผลลัพธ์เปรียบเทียบ ก่อน - หลัง
df_sampled[['Clean_Text', 'Processed_Text', 'Sentiment']].head(3)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


กำลังทำความสะอาดข้อความ...
ทำความสะอาดข้อความเสร็จสิ้น!


,Clean_Text,Processed_Text,Sentiment
0,Outstanding Food!!! This is a very high qualit...,outstanding food high quality dog food meat fr...,1
1,Thank you Betty Crocker I love this cake mix a...,thank betty crocker love cake mix mixes well i...,1
2,Double Black Diamond A nice strong brew. I am ...,double black diamond nice strong brew new keur...,1


|**Feature Extraction**|
|**Feature Selection**|

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Feature Extraction + Feature Selection ในตัวเดียว
tfidf = TfidfVectorizer(
    max_features=5000,  # [Feature Selection] เลือกเฉพาะ 5,000 คำที่มีค่าน้ำหนักสูงสุด
    min_df=5,  # [Feature Selection] ตัดคำที่ปรากฏในรีวิวน้อยกว่า 5 ครั้งทิ้ง
    ngram_range=(
        1,
        2,
    ),  # [Feature Extraction] สกัดทั้งคำเดี่ยวและคำคู่ (เช่น 'not good', 'great taste')
)

# แปลงข้อความให้กลายเป็นตัวเลข
X = tfidf.fit_transform(df_sampled['Processed_Text'])
y = df_sampled['Sentiment']

#  แบ่งข้อมูล Train / Test 80:20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'ขนาดของข้อมูลหลัง Feature Extraction & Selection: {X.shape}')

ขนาดของข้อมูลหลัง Feature Extraction & Selection: (50000, 5000)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# สร้างและฝึกสอนโมเดล Logistic Regression
# class_weight='balanced' จะช่วยชดเชยคลาสน้อย (Negative 16%) อัตโนมัติ
lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

#  ทำนายผลบนชุดข้อมูลทดสอบ (X_test)
y_pred = lr_model.predict(X_test)

#  แสดงรายงานผลการประเมิน
print("=== Classification Report: Sentiment Analysis ===")
print(classification_report(y_test, y_pred, target_names=['Negative (0)', 'Positive (1)']))

#  สกัดคำศัพท์ที่มีอิทธิพลสูงสุดต่อ Sentiment (Model Interpretability)
feature_names = np.array(tfidf.get_feature_names_out())
coefficients = lr_model.coef_[0]

# ดึง 10 คำที่มีค่าน้ำหนักเป็นบวกมากที่สุด (ผลักไปทาง Positive)
top_positive_words = feature_names[np.argsort(coefficients)[-10:]]

# ดึง 10 คำที่มีค่าน้ำหนักเป็นลบมากที่สุด (ผลักไปทาง Negative)
top_negative_words = feature_names[np.argsort(coefficients)[:10]]

print("\n--- 10 คำบ่งชี้รีวิวเชิงบวก (Top Positive Words) ---")
print(list(top_positive_words[::-1]))

print("\n--- 10 คำบ่งชี้รีวิวเชิงลบ (Top Negative Words) ---")
print(list(top_negative_words))

=== Classification Report: Sentiment Analysis ===
              precision    recall  f1-score   support

Negative (0)       0.68      0.89      0.77      1571
Positive (1)       0.98      0.92      0.95      8429

    accuracy                           0.92     10000
   macro avg       0.83      0.91      0.86     10000
weighted avg       0.93      0.92      0.92     10000


--- 10 คำบ่งชี้รีวิวเชิงบวก (Top Positive Words) ---
['great', 'best', 'good', 'delicious', 'excellent', 'love', 'loves', 'perfect', 'yummy', 'nice']

--- 10 คำบ่งชี้รีวิวเชิงลบ (Top Negative Words) ---
['not', 'not good', 'disappointed', 'worst', 'disappointing', 'awful', 'terrible', 'horrible', 'not worth', 'unfortunately']


In [ ]:
import time
import pandas as pd
#โมเดลเพิ่มเติม
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score

#  กำหนดโมเดลทั้งหมดที่ต้องการประชัน
scale_weight = (y_train == 0).sum() / (y_train == 1).sum()  # สำหรับ XGBoost

models = {
    "Logistic Regression (Balanced)": LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    "LinearSVC (Balanced)": LinearSVC(class_weight='balanced', max_iter=2000, random_state=42),
    "Complement Naive Bayes": ComplementNB(),
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=6, scale_pos_weight=scale_weight, random_state=42, n_jobs=-1, eval_metric='logloss')
}

# ลูปเทรนและเก็บสถิติผลลัพธ์
results = []

print("กำลังเริ่มประชันโมเดลทั้งหมด...\n")

for name, model in models.items():
    start_time = time.time()

    # เทรนโมเดล
    model.fit(X_train, y_train)
    train_time = time.time() - start_time

    # พยากรณ์ผล
    y_pred = model.predict(X_test)

    # สกัดตัวชี้วัด
    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    neg_prec = precision_score(y_test, y_pred, pos_label=0)
    neg_recall = recall_score(y_test, y_pred, pos_label=0)
    neg_f1 = f1_score(y_test, y_pred, pos_label=0)

    results.append({
        "Model": name,
        "Accuracy": f"{acc * 100:.2f}%",
        "Macro F1": f"{macro_f1:.4f}",
        "Neg Precision": f"{neg_prec:.4f}",
        "Neg Recall": f"{neg_recall:.4f}",
        "Neg F1-Score": f"{neg_f1:.4f}",
        "Train Time (s)": f"{train_time:.2f}"
    })

#  แสดงตารางสรุปผล
df_results = pd.DataFrame(results)
print("=== ตารางเปรียบเทียบประสิทธิภาพโมเดล (Benchmark) ===")
print(df_results.to_string(index=False))

กำลังเริ่มประชันโมเดลทั้งหมด...

=== ตารางเปรียบเทียบประสิทธิภาพโมเดล (Benchmark) ===
                         Model Accuracy Macro F1 Neg Precision Neg Recall Neg F1-Score Train Time (s)
Logistic Regression (Balanced)   91.84%   0.8622        0.6848     0.8905       0.7742           0.45
          LinearSVC (Balanced)   92.52%   0.8702        0.7142     0.8733       0.7858           1.00
        Complement Naive Bayes   88.51%   0.8185        0.5890     0.8892       0.7086           0.02
                       XGBoost   90.02%   0.8347        0.6343     0.8612       0.7306          59.62


* **LinearSVC (Balanced)**  
  **เหตุผล:** ตัวแทนสาย Support Vector Machine ที่เด่นเรื่องการตัดเส้นแบ่งระนาบ (Hyperplane) บนข้อมูลมิติสูงที่มีค่าศูนย์เป็นส่วนใหญ่ (Sparse Data) ช่วยดันค่า Precision และลดการทายผิดพลาด (False Positives) ได้ดี

* **Logistic Regression (Balanced)**  
  **เหตุผล:** ใช้เป็น Baseline มาตรฐาน ของงานจำแนกข้อความ ประมวลผลเร็ว ตีความค่าน้ำหนักของคำศัพท์ง่าย และมีพารามิเตอร์ถ่วงน้ำหนักชดเชยคลาสน้อย

* **Complement Naive Bayes**  
  **เหตุผล:** ตัวแทนสายความน่าจะเป็น (Probabilistic Model) ที่พัฒนาขึ้นมาแก้จุดบกพร่องของ Naive Bayes แบบดั้งเดิม เพื่อใช้กับข้อมูลข้อความที่ไม่สมดุลโดยเฉพาะ และกินทรัพยากรการคำนวณน้อยที่สุด

* **XGBoost**  
  **เหตุผล:** เพื่อทดสอบสมมติฐานว่าอัลกอริทึมที่จับความสัมพันธ์ของคำแบบไม่เป็นเส้นตรง (Non-linear Interactions) จะทำผลงานได้ดีกว่าโมเดลแบบเส้นตรงหรือไม่

In [ ]:
# ดึงโมเดล LinearSVC ตัวที่เก่งที่สุดออกมาใช้งาน
best_model = models["LinearSVC (Balanced)"]

def predict_review_sentiment(review_text):
    #  ทำความสะอาดข้อความด้วยฟังก์ชันเดิม
    cleaned = clean_review_text(review_text)

    #  แปลงเป็นเวกเตอร์ TF-IDF
    vectorized = tfidf.transform([cleaned])

    #  พยากรณ์ผล (1 = Positive, 0 = Negative)
    prediction = best_model.predict(vectorized)[0]

    label = "Positive (เชิงบวก) " if prediction == 1 else "Negative (เชิงลบ) "
    return label

# ทดสอบกับข้อความรีวิวจริงรูปแบบต่างๆ
sample_reviews = [
    "This coffee is amazing! The flavor is rich and absolutely delicious.",
    "Terrible experience. The package arrived damaged and it tastes awful.",
    "Not good at all, completely disappointed with the quality.",
    "It works okay, but definitely not worth the price."
]

print("=== ผลการทดสอบทำนายข้อความรีวิวสด ===")
for text in sample_reviews:
    print(f"รีวิว: \"{text}\"")
    print(f"ผลทำนาย: {predict_review_sentiment(text)}\n")

=== ผลการทดสอบทำนายข้อความรีวิวสด ===
รีวิว: "This coffee is amazing! The flavor is rich and absolutely delicious."
ผลทำนาย: Positive (เชิงบวก) 

รีวิว: "Terrible experience. The package arrived damaged and it tastes awful."
ผลทำนาย: Negative (เชิงลบ) 

รีวิว: "Not good at all, completely disappointed with the quality."
ผลทำนาย: Negative (เชิงลบ) 

รีวิว: "It works okay, but definitely not worth the price."
ผลทำนาย: Negative (เชิงลบ) 

